In [ ]:
# General notebook settings
import logging
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pypsa

warnings.filterwarnings("error", category=DeprecationWarning)
logging.getLogger("gurobipy").propagate = False
pypsa.options.params.optimize.log_to_console = False

# Subsidies, CfDs and PPAs as Post-Processing

Support schemes such as **feed-in premiums**, **Contracts for Difference (CfDs)** and
**Power Purchase Agreements (PPAs)** rarely change the *dispatch* of an asset: they
redistribute the *revenue* that the asset earns once the market has cleared. That makes
them a natural fit for a **post-processing** step on a solved network, rather than a new
term in the optimisation objective.

This example shows how to settle a renewable generator under several real-world schemes
using only the dispatch (`n.generators_t.p`) and the locational marginal price
(`n.buses_t.marginal_price`) from a standard linear optimal power flow. No custom
constraints or extra variables are required.

The schemes covered:

1. **Two-sided CfD** (UK / EU style): symmetric top-up and clawback around a strike price.
2. **One-sided feed-in premium** with a price floor (Dutch SDE++ style): subsidy only, no clawback.
3. **Cap-and-floor**: revenue stabilised within a price band.
4. **Virtual PPA**: a fixed-price financial swap, contrasted with a baseload PPA.
5. **Advanced variant**: the UK negative-price rule, which pauses CfD payments during
   sustained negative prices, to show how scheme-specific tweaks plug in.

!!! note "Why post-processing?"
    Explicitly modelling a CfD inside the LP would couple revenue to the endogenous price
    $\lambda_t$, which only exists *after* the problem is solved. The standard workaround
    (used here) is to solve a normal LOPF, read out the prices, and settle the contract
    as an exogenous cash flow. See
    [PyPSA issue #1475](https://github.com/PyPSA/PyPSA/issues/1475) for the discussion.

## A Minimal Market

We build a single bidding zone over one week (168 hourly snapshots). Demand follows a
diurnal pattern, and the merit order is set by a small wind fleet plus a ladder of
thermal generators. The wind generator is the asset whose support contract we will settle.

To make the settlement interesting we engineer two features into the week:

- A stretch of **negative prices** during a stormy, low-demand night, where wind is
  curtailed and sets the marginal price (it bids slightly below zero, as subsidised
  renewables do in practice).
- A few **scarcity hours** where the peaking plant sets a price well above the strike,
  triggering clawback under the two-sided schemes.

In [ ]:
n_hours = 168
snapshots = pd.date_range("2030-01-01", periods=n_hours, freq="h")
hour = np.arange(n_hours)

# Diurnal demand: night trough ~160 MW, evening peak ~440 MW.
load = pd.Series(300 + 140 * np.sin(2 * np.pi * (hour - 9) / 24), index=snapshots)

# Wind capacity factor: a weekly swell with a diurnal ripple.
cf = np.clip(
    0.45 + 0.40 * np.sin(2 * np.pi * hour / 47) + 0.12 * np.sin(2 * np.pi * hour / 9),
    0.02,
    1.0,
)
cf = pd.Series(cf, index=snapshots)

# A sustained stormy, low-demand spell (hours 30-41): abundant wind meets low load.
storm = slice(30, 42)
cf.iloc[storm] = 1.0
load.iloc[storm] = 160.0

In [ ]:
P_NOM_WIND = 300.0

n = pypsa.Network()
n.set_snapshots(snapshots)
n.add("Bus", "market")

for carrier in ["wind", "ccgt_base", "ccgt_mid", "ccgt_high", "ocgt_peak"]:
    n.add("Carrier", carrier)

# Wind bids slightly negative (subsidised feed-in), so it sets a negative price when curtailed.
n.add(
    "Generator",
    "wind",
    bus="market",
    carrier="wind",
    p_nom=P_NOM_WIND,
    marginal_cost=-3.0,
    p_max_pu=cf,
)

# Thermal merit-order ladder.
n.add(
    "Generator",
    "ccgt_base",
    bus="market",
    carrier="ccgt_base",
    p_nom=200,
    marginal_cost=30.0,
)
n.add(
    "Generator",
    "ccgt_mid",
    bus="market",
    carrier="ccgt_mid",
    p_nom=180,
    marginal_cost=70.0,
)
n.add(
    "Generator",
    "ccgt_high",
    bus="market",
    carrier="ccgt_high",
    p_nom=150,
    marginal_cost=110.0,
)
n.add(
    "Generator",
    "ocgt_peak",
    bus="market",
    carrier="ocgt_peak",
    p_nom=300,
    marginal_cost=180.0,
)

n.add("Load", "demand", bus="market", p_set=load)

n.optimize(solver_name="highs")

## Reading Out Price and Dispatch

The two ingredients for any settlement are the **market price** the asset sees (here the
locational marginal price at its bus) and its **dispatched energy**. We weight by the
snapshot durations via `n.snapshot_weightings.generators` so the recipe stays correct for
sub-hourly or otherwise weighted snapshots.

In [ ]:
price = n.buses_t.marginal_price["market"]  # EUR/MWh, per snapshot
dispatch = n.generators_t.p["wind"]  # MW, per snapshot
weights = n.snapshot_weightings.generators  # h, per snapshot

energy = dispatch * weights  # MWh per snapshot
total_mwh = energy.sum()
market_revenue = (price * energy).sum()  # EUR earned on the spot market

print(f"Wind energy:          {total_mwh / 1e3:7.1f} GWh")
print(f"Capture price:        {market_revenue / total_mwh:7.1f} EUR/MWh")
print(f"Hours with price < 0: {(price < 0).sum():7d}")
print(f"Price range:          {price.min():.1f} to {price.max():.1f} EUR/MWh")

The **capture price** (revenue divided by generation) is below the time-average price
because wind earns less in the hours when it produces most: the classic cannibalisation
effect. Support schemes exist precisely to insulate the investor from this and from
price volatility.

## A Tiny Settlement Helper

Every scheme below boils down to a **per-snapshot top-up rate** $r_t$ in EUR/MWh applied
to the dispatched energy. The cash flow to the generator in each snapshot is

$$ \text{cashflow}_t = r_t \cdot g_t \cdot w_t $$

where $g_t$ is the dispatch (MW) and $w_t$ the snapshot weighting (h). A positive cash
flow means the generator *receives* money; a negative one means it *pays back* (clawback).
Only the definition of $r_t$ changes between schemes.

In [ ]:
def settle(
    top_up_rate: pd.Series, dispatch: pd.Series, weights: pd.Series
) -> pd.Series:
    """Per-snapshot cash flow (EUR) to the generator for a given top-up rate (EUR/MWh)."""
    return top_up_rate * dispatch * weights


def summarise(name: str, cashflow: pd.Series) -> dict:
    """Headline figures for a settled scheme."""
    subsidy = cashflow.clip(lower=0).sum()
    clawback = abs(cashflow.clip(upper=0).sum())
    net = cashflow.sum()
    achieved = (market_revenue + net) / total_mwh
    return {
        "scheme": name,
        "subsidy_keur": subsidy / 1e3,
        "clawback_keur": clawback / 1e3,
        "net_support_keur": net / 1e3,
        "achieved_price": achieved,
        "subsidy_hours": float((cashflow > 0).mul(weights).sum()),
        "clawback_hours": float((cashflow < 0).mul(weights).sum()),
    }


results = {}

## 1. Two-Sided CfD

The Contract for Difference settles the gap between a fixed **strike price** $S$ and the
market price for every MWh produced:

$$ r_t = S - \lambda_t $$

When the market price is below the strike, the generator is topped up; when it is above,
the generator pays the difference back. Over time the asset earns close to a flat $S$
per MWh regardless of the market, which is exactly what de-risks the investment.

In [ ]:
STRIKE = 65.0  # EUR/MWh

cfd_two_sided = settle(STRIKE - price, dispatch, weights)
results["1. Two-sided CfD"] = summarise("1. Two-sided CfD", cfd_two_sided)
pd.Series(results["1. Two-sided CfD"]).to_frame("value")

## 2. One-Sided Feed-in Premium with a Price Floor (SDE++ style)

The Dutch SDE++ scheme is a **one-sided** subsidy: the state tops the producer up to the
strike (the *basisbedrag*) but never claws back when prices are high. Crucially it also
applies a **price floor**: the subsidy is computed against `max(price, floor)`, so once
the market price drops below the floor (and especially when it goes negative) the
top-up stops growing. This deliberately exposes the producer to deep negative-price
hours, discouraging production into a saturated market.

$$ r_t = \max\bigl(S - \max(\lambda_t,\ \text{floor}),\ 0\bigr) $$

In [ ]:
FLOOR = 0.0  # EUR/MWh: subsidy frozen at/below this market price

premium_rate = np.maximum(STRIKE - price.clip(lower=FLOOR), 0.0)
fip_one_sided = settle(premium_rate, dispatch, weights)
results["2. One-sided premium"] = summarise("2. One-sided premium", fip_one_sided)
pd.Series(results["2. One-sided premium"]).to_frame("value")

Note the absence of clawback hours, and that the net support is **larger** than the
two-sided CfD: there is no payback in the scarcity hours. The price floor caps how much
the scheme pays in the cheapest hours.

## 3. Cap-and-Floor

A cap-and-floor scheme (used e.g. for interconnectors and some merchant assets)
stabilises revenue within a band. The producer keeps its market revenue, but is topped
up when the price falls below a floor $F$ and pays back when it rises above a cap $C$:

$$ r_t = \max(F - \lambda_t,\ 0) - \max(\lambda_t - C,\ 0) $$

Inside the band $[F, C]$ the producer is fully exposed to the market.

In [ ]:
PRICE_FLOOR, PRICE_CAP = 40.0, 120.0

caf_rate = np.maximum(PRICE_FLOOR - price, 0.0) - np.maximum(price - PRICE_CAP, 0.0)
cap_and_floor = settle(caf_rate, dispatch, weights)
results["3. Cap-and-floor"] = summarise("3. Cap-and-floor", cap_and_floor)
pd.Series(results["3. Cap-and-floor"]).to_frame("value")

## 4. Virtual PPA vs Baseload PPA

A **virtual (financial) PPA** is a fixed-price swap on the asset's *generated* volume.
At a contract price $P$ the off-taker pays the generator $P$ per MWh and keeps the spot
revenue, so the generator's net contract cash flow is identical in shape to a two-sided
CfD with strike $P$:

$$ r^{\text{gen}}_t = P - \lambda_t \qquad r^{\text{buyer}}_t = \lambda_t - P $$

The two parties' cash flows are mirror images and sum to zero.

In [ ]:
PPA_PRICE = 60.0  # EUR/MWh

vppa_generator = settle(PPA_PRICE - price, dispatch, weights)
vppa_buyer = settle(price - PPA_PRICE, dispatch, weights)
results["4a. vPPA (generator)"] = summarise("4a. vPPA (generator)", vppa_generator)

print(f"vPPA generator net: {vppa_generator.sum() / 1e3:8.1f} kEUR")
print(f"vPPA buyer net:     {vppa_buyer.sum() / 1e3:8.1f} kEUR")
print(f"Sum (should be ~0): {(vppa_generator + vppa_buyer).sum():8.2e} EUR")

A **baseload PPA** instead fixes a constant volume $Q$ every hour, independent of when
the asset actually generates. The buyer settles that flat block against the spot price:

$$ \text{cashflow}^{\text{buyer}}_t = (\lambda_t - P) \cdot Q \cdot w_t $$

Because the block does not follow the wind, the buyer carries **shape (profile) risk**:
it buys at the flat price even in hours when the asset produces nothing. We compare the
buyer's exposure under both PPA structures.

In [ ]:
Q_BASELOAD = (
    total_mwh / weights.sum()
)  # MW: flat block sized to the asset's average output
baseload_buyer = settle(
    price - PPA_PRICE, pd.Series(Q_BASELOAD, index=snapshots), weights
)

print(f"Flat baseload block:        {Q_BASELOAD:6.1f} MW")
print(f"vPPA buyer net:             {vppa_buyer.sum() / 1e3:8.1f} kEUR")
print(f"Baseload PPA buyer net:     {baseload_buyer.sum() / 1e3:8.1f} kEUR")
print(
    f"Shape-risk gap:             {(baseload_buyer.sum() - vppa_buyer.sum()) / 1e3:8.1f} kEUR"
)

## 5. Advanced: the UK Negative-Price Rule

Recent UK CfD allocation rounds withhold payments during **sustained negative prices**:
no top-up is paid for any hour belonging to a run of (here) **6 or more** consecutive
negative-price hours. This discourages renewables from running when the system is
already oversupplied. It is a good illustration of how a scheme-specific rule slots into
the post-processing step as a simple boolean mask.

In [ ]:
NEG_RUN_HOURS = 6

is_negative = price < 0
run_id = (is_negative != is_negative.shift()).cumsum()  # contiguous-run labels
run_length = is_negative.groupby(run_id).transform("size")  # length of each run
excluded = is_negative & (run_length >= NEG_RUN_HOURS)  # hours where payment pauses

cfd_neg_rule = cfd_two_sided.where(~excluded, 0.0)
results["5. Two-sided CfD + neg-rule"] = summarise(
    "5. Two-sided CfD + neg-rule", cfd_neg_rule
)

print(f"Negative-price hours:                 {is_negative.sum():4d}")
print(f"Hours in runs >= {NEG_RUN_HOURS}h (payment paused): {int(excluded.sum()):4d}")
print(
    f"Top-up withheld vs plain CfD:         "
    f"{(cfd_two_sided[excluded].sum()) / 1e3:8.1f} kEUR"
)

## Comparison Across Schemes

Pulling the headline figures together. `achieved_price` is the asset's effective
realised price (market capture price plus net support, per MWh); the schemes differ
mainly in how much risk they leave with the producer versus the counterparty.

In [ ]:
comparison = pd.DataFrame(results.values()).set_index("scheme").round(1)
comparison

## Visualising the Settlement

The first plot shows the market price against the CfD strike over the first three days,
with the negative-price run that triggers the UK rule shaded. The second compares the
net support and the achieved price across all schemes.

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(10, 8), gridspec_kw={"height_ratios": [3, 2]}
)

window = slice(0, 72)
ax1.step(
    price.index[window],
    price.iloc[window],
    where="post",
    color="C0",
    label="Market price",
)
ax1.axhline(STRIKE, color="C3", ls="--", label=f"CfD strike ({STRIKE:.0f})")
ax1.axhline(0, color="grey", lw=0.8)
ax1.fill_between(
    price.index[window],
    0,
    price.iloc[window],
    where=(price.iloc[window] < 0),
    step="post",
    color="C0",
    alpha=0.25,
    label="Negative prices",
)
ax1b = ax1.twinx()
ax1b.fill_between(
    dispatch.index[window],
    0,
    dispatch.iloc[window],
    step="post",
    color="C2",
    alpha=0.15,
)
ax1b.set_ylabel("Wind dispatch [MW]", color="C2")
ax1b.tick_params(axis="y", labelcolor="C2")
ax1.set_ylabel("Price [EUR/MWh]")
ax1.set_title("Market price vs CfD strike (first 3 days)")
ax1.legend(loc="upper left", fontsize=8)

net = comparison["net_support_keur"]
colors = ["C3" if v < 0 else "C0" for v in net]
ax2.bar(range(len(net)), net.values, color=colors, alpha=0.8)
ax2.set_xticks(range(len(net)))
ax2.set_xticklabels(comparison.index, rotation=20, ha="right", fontsize=8)
ax2.axhline(0, color="grey", lw=0.8)
ax2.set_ylabel("Net support [kEUR]")
ax2.set_title("Net support to the generator by scheme")

plt.tight_layout()
plt.show()

## Takeaways

- CfDs, feed-in premiums and PPAs can all be settled as a **post-processing step** on a
  solved network: read `n.buses_t.marginal_price` and `n.generators_t.p`, define a
  per-snapshot top-up rate, and weight by `n.snapshot_weightings.generators`.
- The schemes share one line of arithmetic; what differs is whether they claw back
  (two-sided CfD, cap-and-floor), apply a floor (SDE++), settle a fixed volume (baseload
  PPA), or mask out specific hours (the negative-price rule).
- Because the contract does not enter the LP, the dispatch is unchanged. This is the
  standard approximation; modelling the price feedback explicitly would require a
  bilevel / MPEC formulation, which is out of scope here.

These recipes are deliberately small and self-contained so they can be adapted to a
specific scheme. See [issue #1475](https://github.com/PyPSA/PyPSA/issues/1475) for the
wider discussion on turning post-processing steps like these into reusable plugins.